# Suite SYN — Syntax trees

Every language's trees implement the same protocols (`@mbse/programs/Framework/Syntax`): kinds declared once in
their specs, nodes that serialize through derived meta-schemas and builders, validation, standards, and traversal by
fields alone. This suite checks that machinery on a small language declared here, Toy, and on Ccpp where real trees
help.

In [ ]:
import * as Programs from "@mbse/programs";
import { Syntax as S } from "@mbse/programs/Ccpp";
import { Syntax as F } from "@mbse/programs/Framework";
import { Errors, JSON as SchemaJSON, Proxies } from "@mbse/schemas/Framework";

import { assert, equal, raises } from "./support.js";

const { AttributeError, KeyError, LookupError, NotImplementedError, ValueError } = Errors;
const { Language, Node, choice, flag, many, one, optional, optionalChoice, optionalInteger, text } = F;

abstract class Toy extends Node {}

const LeafSpec = { value: text(), flag: flag(), count: optionalInteger(), color: optionalChoice("red", "blue") };
interface Leaf extends F.Fields<typeof LeafSpec> {}
class Leaf extends Toy {
  static override SPEC = LeafSpec;
  static override SINCE = { Toy: 2000 };
  static override FEATURES: F.Features = { color: [["blue", { Toy: 2010 }]], flag: [[true, { Toy: [2000, 2020] }]] };
}

const PairSpec: {
  left: F.ChildSpec<Toy, false, false>; right: F.ChildSpec<Toy, false, true>; items: F.ChildSpec<Toy, true, true>;
} = { left: one(() => [Toy]), right: optional(() => [Toy]), items: many(() => [Toy]) };
interface Pair extends F.Fields<typeof PairSpec> {}
class Pair extends Toy {
  static override SPEC = PairSpec;
  override features(): [string, F.Availability][] {
    return this.items.length === 3 ? [["Pair with three items", { Toy: 2030 }]] : [];
  }
  override check(): string[] {
    return this.left === this ? ["a Pair cannot hold itself"] : [];
  }
}

const ExtraSpec = { leaf: optional(() => [Leaf]) };
interface Extra extends F.Fields<typeof ExtraSpec> {}
class Extra extends Toy {
  static override SPEC = ExtraSpec;
  static override EXTENSION = true;
}

const TOY = new Language("Toy", [Leaf, Pair, Extra], { base: { Toy: 1990 } });
const leaf = (value = "a", fields: object = {}) => new Leaf({ value, ...fields });
const pair = (fields: object) => new Pair(fields);
const values = (nodes: Leaf[]) => nodes.map((n) => n.value);
const native = (value: unknown) => (p: any) => p.value((a: any) => a.as_native((n: any) => n.set(value)));

## SYN-01 · Kinds declare their fields in their specs

In [ ]:
equal(Leaf.FIELDS.map((f) => f.name), ["value", "flag", "count", "color"]);
equal(Leaf.FIELDS.map((f) => f.describe()), [
  { name: "value", type: "str", optional: false }, { name: "flag", type: "bool", optional: false },
  { name: "count", type: "int", optional: true },
  { name: "color", type: "str", optional: true, choices: ["red", "blue"] }]);
equal(Pair.FIELDS.map((f) => f.describe()), [
  { name: "left", type: ["Toy"], optional: false, many: false },
  { name: "right", type: ["Toy"], optional: true, many: false },
  { name: "items", type: ["Toy"], optional: true, many: true }]);
assert(Leaf.KIND === "Leaf" && Leaf.NAME === "Programs.Toy.Leaf" && Leaf.LANGUAGE === TOY);
const condition = S.IfStatement.FIELDS[4] as F.Child;
assert(condition.name === "condition" && condition.optional && !condition.many);
equal(condition.categories, [S.Expression, S.Declaration]);
assert(Pair.BY_NAME.get("items") instanceof F.Child && Leaf.BY_NAME.get("value") instanceof F.Attribute);
equal([text(), flag(), choice("a")].map((s) => [s.field, s.optional]), [["attribute", false], ["attribute", false],
  ["attribute", false]]);
console.log("ok");

## SYN-02 · Nodes are plain objects: construction, defaults, `toString` (Python's `repr`), identity

In [ ]:
let made = leaf("x", { flag: true, count: 3n });
equal([made.value, made.flag, made.count, made.color], ["x", true, 3n, null]);
const p1 = pair({ left: made, items: new Set([leaf("y")]) });
assert(p1.right === null && Array.isArray(p1.items) && pair({ left: made }).items.length === 0);
assert(String(p1) === "Pair(left=Leaf(value='x', flag=True, count=3), items=[Leaf(value='y')])");
assert(String(leaf("z", { count: 0n })) === "Leaf(value='z', count=0)");
assert(new Leaf().flag === false && new Leaf().value === null);
raises(TypeError, () => new (Toy as unknown as new () => Node)(), "Toy is not a kind of node");
raises(TypeError, () => new Leaf({ colour: "red" }), "Leaf has no field 'colour'");
assert(typeof made.identity() === "number" && made.identity() !== leaf().identity());
assert(made.schema_name() === "Programs.Toy.Leaf" && made.owner() === null && made.kind() === Leaf);
equal([made.check(), made.features(), made.availability()], [[], [], { Toy: 2000 }]);
const listed = F.children(pair({ left: made, items: [null, made] }));
assert(listed.length === 2 && listed[0]?.[2] === made && listed[1]?.[2] === made);
equal(listed.map(([f, i]) => [f, i]), [["left", null], ["items", 1]]);
const emptied = pair({ left: made });
(emptied as any).items = null;
assert(F.children(emptied).length === 1 && Programs.Ccpp.Syntax === S);
console.log("ok");

## SYN-03 · Meta-schemas, registration and snapshots

In [ ]:
assert(Proxies.schema("Programs.Toy.Leaf") === Leaf.Schema);
equal([...Leaf.Schema.properties.keys()], ["kind", "value", "flag", "count", "color"]);
equal([...Leaf.Schema.adjacencies.keys()], ["parent"]);
equal([...Pair.Schema.adjacencies.keys()], ["children", "parent"]);
equal(TOY.Schema.branches.map((b) => b.name), ["Leaf", "Pair", "Extra"]);
let tree = pair({ left: leaf("x", { flag: true }), right: leaf("y", { color: "blue", count: 0n }), items: [leaf("z"), leaf("w")] });
const snapshot = SchemaJSON.ToJSON.Reachable(Pair.Schema, tree);
const plain = JSON.parse(snapshot);
equal(plain.objects.s0, { kind: "Pair", children: [
  { child: { $ref: "s1", $schema: "Programs.Toy.Leaf" }, field: "left" },
  { child: { $ref: "s2", $schema: "Programs.Toy.Leaf" }, field: "right" },
  { child: { $ref: "s3", $schema: "Programs.Toy.Leaf" }, field: "items", index: 0 },
  { child: { $ref: "s4", $schema: "Programs.Toy.Leaf" }, field: "items", index: 1 }] });
equal(plain.objects.s1, { kind: "Leaf", value: "x", flag: true }); // false is not written
equal(plain.objects.s2, { kind: "Leaf", value: "y", count: 0, color: "blue" });
const copied0 = SchemaJSON.FromJSON(TOY.Builders).Reachable(Pair.Schema, snapshot) as Pair;
assert(F.same(copied0, tree) && copied0 !== tree && (copied0.left as Leaf).flag === true && (copied0.right as Leaf).flag === false);
const unit = new S.TranslationUnit({ items: [new S.ExpressionStatement({ expression: new S.IntegerLiteral({ spelling: "1" }) })] });
assert(F.same(SchemaJSON.FromJSON(S.LANGUAGE.Builders).Reachable(S.TranslationUnit.Schema,
  SchemaJSON.ToJSON.Reachable(S.TranslationUnit.Schema, unit)), unit));
console.log("ok");

## SYN-04 · Builders: the DSL, `create`/`clone`/`update`, and their errors

In [ ]:
made = TOY.builder("Pair").set("left", leaf("x")).add("items", leaf("y")).add("items", leaf("z")).create();
const madePair = made as unknown as Pair;
equal(values(madePair.items as Leaf[]), ["y", "z"]);
assert((madePair.left as Leaf).value === "x");
const again = TOY.builder(Pair, madePair).set("items", [leaf("q")]).set("right", leaf("r")).clone() as Pair;
assert(again !== madePair && values(again.items as Leaf[]).join() === "q" && madePair.right === null);
const updated = TOY.builder(Pair, madePair).set("right", null).set("left", leaf("l")).update();
assert(updated === madePair && (madePair.left as Leaf).value === "l");
assert(TOY.builder("Leaf").set("value", "v").set("flag", true).create().flag === true);
raises(KeyError, () => TOY.builder("Tree"), "Toy has no kind 'Tree'");
raises(TypeError, () => TOY.builder(Leaf, madePair), "expected a Leaf to build from, got Pair");
raises(KeyError, () => TOY.builder(Pair).set("middle", leaf()), "Pair has no child field 'middle'");
raises(TypeError, () => TOY.builder(Pair).add("left", leaf()), "Pair.left holds one child; use set()");
raises(ValueError, () => TOY.builder(Pair, madePair).create(), "create() is only valid without a source instance");
raises(ValueError, () => TOY.builder(Pair).clone(), "clone() is only valid with a source instance");
raises(ValueError, () => TOY.builder(Pair).update(), "update() is only valid with a source instance");

/** A Pair built from children entries given as [link target, field, index], through the visitor protocol. */
function built(...entries: [unknown, string | null, bigint | null][]): Pair {
  const b = TOY.builder(Pair);
  for (const [target, field, index] of entries) {
    b.adjacency("children", (a) => a.add((entry) => {
      if (target !== null) entry.link("child", (k) => k.set(target as any));
      if (field !== null) entry.property("field", native(field));
      if (index !== null) entry.property("index", native(index));
    }));
  }
  return b.create();
}

let [x, y, z] = [leaf("x"), leaf("y"), leaf("z")];
equal(values(built([z, "items", null], [y, "items", 1n], [x, "items", 0n], [x, "left", null]).items as Leaf[]), ["x", "y", "z"]);
equal(values(built([z, "items", null], [y, "items", null], [x, "left", null]).items as Leaf[]), ["z", "y"]);
raises(ValueError, () => built([null, "left", null]), "link 'child' is not set");
raises(ValueError, () => built([x, null, null]), "a child entry needs a field");
raises(ValueError, () => built([x, "middle", null]), "a Pair has no child field 'middle'");
raises(TypeError, () => built(["x", "left", null]), "a child must be a node, got str");
raises(ValueError, () => built([x, "left", 0n]), "Pair.left holds one child, not a list");
raises(ValueError, () => built([x, "left", null], [y, "left", null]), "Pair.left holds one child, got several");
console.log("ok");

## SYN-05 · Builders, entries and links implement the visitor protocols

In [ ]:
const b = TOY.builder(Leaf, leaf("v", { count: 2n }));
const names: [string, boolean][] = [];
b.properties((p) => names.push([p.name(), p.has()]));
equal(names, [["kind", true], ["value", true], ["count", true]]);
assert(b.has("kind") && !b.has("flag") && !b.has("nothing"));
b.property("flag", native(true));
assert(b.has("flag"));
const read: unknown[] = [];
b.property("value", (p) => p.value((a) => a.as_native((n) => { read.push(n.get()); })));
equal(read, ["v"]);
b.property("kind", native("Leaf"));
raises(ValueError, () => b.property("kind", native("Pair")), "expected kind 'Leaf', got 'Pair'");
raises(TypeError, () => b.property("count", native("2")), "expected int, got str");
raises(KeyError, () => b.property("size", () => null), "unknown property 'size'");
b.clear("count").clear("kind").clear("flag").clear("nothing");
assert(!b.has("count") && b.has("kind") && !b.has("flag"));
raises(AttributeError, () => b.property("count", (p) => p.value((a) => a.as_native((n) => n.get()))), "property 'count' is not set");
for (const kind of ["as_object", "as_union", "as_intersection", "as_indexed"]) {
  raises(TypeError, () => b.property("value", (p) => p.value((a: any) => a[kind](() => null))), "property 'value' is native");
}
assert(b.identify(madePair) === b);
const seen: unknown[] = [];
b.adjacencies((a) => seen.push([a.name(), a.me()]));
equal(seen, [["parent", "child"]]);
raises(KeyError, () => b.adjacency("children", () => null), "unknown adjacency 'children'");
b.adjacency("parent", (a) => a.add(() => null).entries((e) => seen.push(e)).remove(null as any));
assert(seen.length === 1); // `parent` holds no entries: parents' children imply them

const pb = TOY.builder(Pair, pair({ left: x, items: [y] }));
const adjacencies: string[] = [];
pb.adjacencies((a) => adjacencies.push(a.name()));
equal(adjacencies, ["children", "parent"]);
const entries: any[] = [];
pb.adjacency("children", (a) => a.entries((e) => entries.push(e)));
const first = entries[0];
const links: unknown[] = [];
const fields: unknown[] = [];
first.links((k: any) => links.push(k.name()));
first.link("child", (k: any) => k.target((t: unknown) => links.push(t)));
first.properties((q: any) => fields.push([q.name(), q.has()]));
assert(links[0] === "child" && links[1] === x);
equal(fields, [["field", true]]);
const indexed: unknown[] = [];
entries[1].properties((q: any) => indexed.push([q.name(), q.has()]));
equal(indexed, [["field", true], ["index", true]]);
entries[1].clear("index");
assert(!entries[1].has("index"));
assert(first.has("field") && !first.has("index") && !first.has("other"));
raises(KeyError, () => first.link("parent", () => null), "'parent' is not a link this entry can set");
raises(KeyError, () => first.property("other", () => null), "unknown property 'other'");
first.clear("field").clear("other");
assert(!first.has("field"));
pb.adjacency("children", (a) => a.remove(entries[1]));
raises(ValueError, () => pb.clone(), "a child entry needs a field"); // the first entry lost its field
first.property("field", native("left"));
equal((pb.clone() as Pair).items, []); // and the second was removed
raises(ValueError, () => pb.adjacency("children", (a) => a.add((e) => e.links((k) => k.target(() => null)))),
  "link 'child' is not set");
console.log("ok");

## SYN-06 · The registry builds nodes from snapshots

In [ ]:
const registry = TOY.Builders;
assert(registry.schema("Programs.Toy.Leaf") === Leaf.Schema && registry.name_of(Pair.Schema) === "Programs.Toy.Pair");
assert(registry.member(leaf("m"), "value") === "m");
raises(TypeError, () => registry.schema(F.CHILDREN), "'Programs.Children' is a relation; no relation builder is exposed");
raises(AttributeError, () => registry.schema("Programs.Toy.Tree"), "no schema registered as 'Programs.Toy.Tree'");
raises(LookupError, () => registry.name_of(S.Identifier.Schema), "schema is not registered");
assert(TOY.name() === "Toy" && String(TOY) === "<language Toy>");
equal([...TOY.kinds().keys()], ["Leaf", "Pair", "Extra"]);
equal([...TOY.categories().keys()], ["Toy"]);
assert(TOY.schema_of(leaf()) === Leaf.Schema);
raises(TypeError, () => TOY.schema_of(3n), "not a Toy node: 3");
const grammar = TOY.grammar() as any;
assert(grammar.language === "Toy");
equal(grammar.base, { Toy: 1990 });
equal(grammar.categories, [{ name: "Toy", base: "Node" }]);
equal(grammar.kinds[0], { kind: "Leaf", category: "Toy", since: { Toy: 2000 }, extension: false,
  features: { color: { blue: { Toy: 2010 } }, flag: { true: { Toy: [2000, 2020] } } },
  fields: Leaf.FIELDS.map((f) => f.describe()) });
assert(grammar.kinds[1].since === null && grammar.kinds[2].extension === true);
console.log("ok");

## SYN-07 · Validation: required fields, categories, choices, sharing, cycles and kinds' own checks

In [ ]:
equal(TOY.validate(pair({ left: leaf(), items: [leaf()] })), []);
equal(TOY.validate(pair({})), ["a Pair needs a left"]);
equal(TOY.validate(new Leaf({ count: "3", color: "green" })),
  ["a Leaf needs a value", "Leaf.count must be an int, got str", "Leaf.color cannot be 'green'"]);
equal(TOY.validate(new Leaf({ value: 3n, flag: 1n })), ["Leaf.value must be a str, got int", "Leaf.flag must be a bool, got int"]);
const shared = leaf();
equal(TOY.validate(pair({ left: shared, items: [leaf(), shared] })), ["items[1]: the node is also at left"]);
let cycle = pair({ items: [] });
cycle.left = cycle;
equal(TOY.validate(cycle), ["left: the node is also at the root", "a Pair cannot hold itself"]);
equal(TOY.validate(pair({ left: new S.Identifier({ spelling: "a" }), right: "b", items: [null, 3n] })), [
  "left: Pair.left must be a Toy, got an Identifier", "right: Pair.right must be a Toy, got str",
  "Pair.items[0] is empty", "items[1]: Pair.items must be a Toy, got int"]);
const notAList = pair({ left: leaf() });
(notAList as any).items = leaf();
equal(TOY.validate(notAList), ["Pair.items must be a list, got Leaf"]);
equal(TOY.validate(pair({ left: pair({ left: new Leaf() }) })), ["left.left: a Leaf needs a value"]);
equal(TOY.validate("x"), ["not a Toy node: 'x'"]);
equal(S.LANGUAGE.validate(new S.BinaryExpression({ left: new S.IdExpression({ name: new S.Identifier({ spelling: "9" }) }),
  operator: "+", right: new S.IntegerLiteral({ spelling: "1" }) })), ["left.name: '9' is not an identifier"]);
console.log("ok");

## SYN-08 · Standards: availability of kinds and features, and their problems

In [ ]:
const toy = (family: string, year: number) => new F.Standard(TOY, family, year);
const [toy2015, toy2025] = [toy("Toy", 2015), toy("Toy", 2025)];
assert(toy2015.name() === "Toy15" && String(toy2025) === "<standard Toy25>" && F.label("C++", 1998) === "C++98");
assert(toy2015.version === 2015 && toy2015.label(1998) === "Toy98");
class Numbered extends F.Standard {
  override label(version: number): string {
    return `Toy ${version}`;
  }
}
equal(new Numbered(TOY, "Toy", 1995).problems(leaf()), ["Leaf needs Toy 2000"]);
assert(new Numbered(TOY, "Toy", 2025).name() === "Toy 2025");
assert(toy2015.has({ Toy: 2000 }) && !toy2015.has({ Toy: 2020 }) && !toy2015.has({ Other: 1 }));
assert(toy2015.has({ Toy: [2000, 2020] }) && !toy2025.has({ Toy: [2000, 2020] }));
equal(toy2015.problems(leaf("a", { color: "blue", flag: true })), []);
equal(toy2025.problems(leaf("a", { color: "blue", flag: true })), ["Leaf.flag was removed in Toy20"]);
equal(toy("Toy", 2005).problems(leaf("a", { color: "blue" })), ["Leaf.color 'blue' needs Toy10"]);
equal(toy("Toy", 1995).problems(leaf()), ["Leaf needs Toy00"]);
equal(toy("Other", 2000).problems(leaf()), ["Leaf is not Other"]);
equal(toy("Other", 2000).problems(new Extra()), []); // extensions are in every standard
equal(toy("Other", 2000).problems(pair({})), ["Pair is not Other"]); // by the language's base
const three = pair({ left: leaf(), items: [leaf(), leaf(), leaf("a", { color: "blue" })] });
equal(toy("Toy", 2005).check(three), ["Pair with three items needs Toy30", "items[2]: Leaf.color 'blue' needs Toy10"]);
equal(toy("Toy", 2030).check(three), []);
raises(NotImplementedError, () => toy2015.parse(""), "Toy15 has no parser");
raises(NotImplementedError, () => toy2015.print(leaf()), "Toy15 has no printer");
console.log("ok");

## SYN-09 · `children`, `walk`, `fold`, `same` and `copy`, with sharing and cycles

In [ ]:
[x, y] = [leaf("x"), leaf("y")];
tree = pair({ left: x, right: pair({ left: y, items: [x] }), items: [y, x] });
equal([...F.walk(tree)].map((n) => (n instanceof Leaf ? n.value : "pair")), ["pair", "x", "pair", "y"]);
assert(F.fold<string>(tree, (n, rs) => (n instanceof Leaf ? n.value as string : "(" + rs.join(" ") + ")")) === "(x (y x) y x)");
const calls: Node[] = [];
F.fold(tree, (n) => { calls.push(n); });
assert(calls.length === 4); // shared nodes once
cycle = pair({ items: [] });
cycle.left = cycle;
const walked = [...F.walk(cycle)];
assert(walked.length === 1 && walked[0] === cycle);
raises(ValueError, () => F.fold(cycle, () => null), "the tree contains a cycle");
assert(F.same(tree, F.copy(tree)) && !F.same(tree, pair({ left: x })));
assert(!F.same(leaf("x"), leaf("y")) && !F.same(leaf("x"), pair({})) && !F.same(leaf("a", { count: 1n }), leaf("a", { count: true })));
assert(!F.same(pair({ left: x, items: [x] }), pair({ left: x, items: [y] })) && !F.same(pair({ left: x }), pair({ left: y })));
assert(F.same(null, null) && !F.same(x, null));
const other = pair({ items: [] });
other.left = other;
assert(F.same(cycle, other));
const copied = F.copy(tree);
assert(copied.left === (copied.right as Pair).items[0] && copied.left !== x); // sharing is kept
const loop = F.copy(cycle);
assert(loop.left === loop && loop !== cycle);
console.log("ok");

## SYN-10 · `Parents`: where nodes are, and changing the tree in place

In [ ]:
const [a, bb, c, d] = [leaf("a"), leaf("b"), leaf("c"), leaf("d")];
const inner = pair({ left: c });
tree = pair({ left: a, right: inner, items: [bb, d] });
const parents = new F.Parents(tree);
const at = (node: Node) => parents.location(node) as [Node, string, number | null];
assert(parents.parent(c) === inner && parents.parent(tree) === null && at(d)[0] === tree);
equal(at(d).slice(1), ["items", 1]);
const ancestry = [...parents.ancestors(c)];
assert(ancestry[0] === inner && ancestry[1] === tree && parents.path(c) === "right.left" && parents.path(tree) === "");
assert(parents.path(d) === "items[1]" && parents.location(leaf()) === null);
const [e, f, g] = [leaf("e"), leaf("f"), pair({ left: leaf("h") })];
parents.replace(bb, [e, f]);
equal(values(tree.items as Leaf[]), ["e", "f", "d"]);
equal(at(d).slice(1), ["items", 2]);
parents.replace(a, g);
assert(tree.left === g && parents.path(g.left as Node) === "left.left");
parents.remove(e);
equal(values(tree.items as Leaf[]), ["f", "d"]);
equal(at(d).slice(1), ["items", 1]);
parents.remove(inner);
assert(tree.right === null && parents.parent(inner) === null);
raises(ValueError, () => parents.replace(inner, leaf()), "the node has no parent in this tree");
raises(TypeError, () => parents.replace(g, [leaf()]), "Pair.left holds one node, not a list");
assert(tree.left === g);
console.log("ok");

## SYN-11 · `Visitor` and `Transformer` dispatch by kind

In [ ]:
class Values extends F.Visitor {
  readonly found: unknown[] = [];

  visit_Leaf(node: Leaf): void {
    this.found.push(node.value);
  }
}

const visitor = new Values();
visitor.visit(pair({ left: leaf("a"), right: pair({ left: leaf("b") }), items: [leaf("c")] }));
equal(visitor.found, ["a", "b", "c"]);

class Rewrite extends F.Transformer {
  visit_Leaf(node: Leaf): unknown {
    if (node.value === "drop") return null;
    if (node.value === "twice") return [leaf("one"), leaf("two")];
    if (node.value === "upper") return leaf("UPPER");
    return node;
  }
}

tree = pair({ left: leaf("upper"), right: leaf("drop"), items: [leaf("keep"), leaf("twice"), leaf("drop")] });
assert(new Rewrite().visit(tree) === tree);
assert((tree.left as Leaf).value === "UPPER" && tree.right === null);
equal(values(tree.items as Leaf[]), ["keep", "one", "two"]);
raises(TypeError, () => new Rewrite().visit(pair({ left: leaf("twice") })), "Pair.left holds one node, not a list");

class Rename extends F.Transformer {
  visit_Identifier(node: S.Identifier): S.Identifier {
    return new S.Identifier({ spelling: (node.spelling as string).toUpperCase() });
  }
}

const sum = new S.BinaryExpression({ left: new S.IdExpression({ name: new S.Identifier({ spelling: "a" }) }), operator: "+",
  right: new S.IdExpression({ name: new S.Identifier({ spelling: "b" }) }) });
assert(new Rename().visit(sum) === sum && ((sum.left as S.IdExpression).name as S.Identifier).spelling === "A");
assert(new Rename().visit(new S.ReturnStatement()).value === null);
console.log("ok");